# 07 Final TabM and Tree Stacking

Train TabM, LightGBM, XGBoost and ExtraTrees using chronological OOF predictions, then generate the second final submission. Report: Section 3.4.


## 1. Setup
Set the data path, choose a runtime and install the dependencies used below. The `quick` profile checks the workflow with a small budget; use `report` for the report experiments. Place the four original CSV files in one folder.


In [ ]:
from pathlib import Path
# Use the same data directory and run name in all notebooks.
PROFILE = globals().get('PROFILE', 'report')  # quick: smoke test; report: full training
DATA_SOURCE = globals().get('DATA_SOURCE', 'drive')  # drive, upload or local
DATA_DIR = Path(globals().get('DATA_DIR', '/content/drive/MyDrive/bt-4012-competition-2026'))
OUTPUT_ROOT = Path(globals().get('OUTPUT_ROOT', '/content/drive/MyDrive/BT4012_Final_Report_English'))
RUN_NAME = globals().get('RUN_NAME', 'final_report_english_v1')
INSTALL_DEPENDENCIES = globals().get('INSTALL_DEPENDENCIES', True)
EXPECTED_COMPETITION = globals().get('EXPECTED_COMPETITION', True)
assert PROFILE in ['quick', 'report']
OUT = OUTPUT_ROOT / RUN_NAME / PROFILE


In [ ]:
import sys, subprocess
# Keep the torch/CUDA version supplied by Colab.
PACKAGES = ['numpy>=1.26', 'pandas>=2.0', 'scipy>=1.11', 'scikit-learn>=1.6', 'joblib', 'tabm==0.0.3', 'rtdl-num-embeddings==0.0.12', 'xgboost>=2.0', 'lightgbm>=4.0']
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)


In [ ]:
CSV_FILES = ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif DATA_SOURCE == 'upload':
    from google.colab import files
    DATA_DIR = Path('/content/bt4012_data')
    DATA_DIR.mkdir(exist_ok=True)
    missing = [name for name in CSV_FILES if not (DATA_DIR / name).exists()]
    if missing:
        print('Upload the missing files:', missing)
        for name, content in files.upload().items():
            if name not in missing:
                raise ValueError('Unexpected file: ' + name)
            (DATA_DIR / name).write_bytes(content)
    if str(OUTPUT_ROOT).startswith('/content/drive/'):
        OUTPUT_ROOT = Path('/content/BT4012_Final_Report_English')
        OUT = OUTPUT_ROOT / RUN_NAME / PROFILE
elif DATA_SOURCE != 'local':
    raise ValueError(DATA_SOURCE)
for name in CSV_FILES:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUT.mkdir(parents=True, exist_ok=True)
print('Mode:', PROFILE, '| Output:', OUT)


## 2. Required functions
Only the data structures, models and helpers used by this notebook are included below.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from torch.nn import functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from pathlib import Path
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from scipy.sparse.csgraph import connected_components
from scipy.sparse import coo_matrix
from collections import deque
import gc
import hashlib
import joblib
import json
from torch import nn
import numpy as np
import pandas as pd
import random
from sklearn.metrics import roc_auc_score
import torch
from sklearn.model_selection import train_test_split
import os
torch.set_num_threads(min(4, os.cpu_count() or 1))

ENGINE_HASH = '3661498eb4e7e545253aeae84b480c6d4c349dc7e125920bf97989a5e96c0840'

FEATURES = [f'feat_{i}' for i in range(1, 166)]

TABM = dict(k=32, n_blocks=2, d_block=512, dropout=0.1, n_bins=48, d_embedding=16, learning_rate=0.002, weight_decay=0.0003)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def serial(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, Path):
        return str(x)
    raise TypeError(type(x).__name__)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=serial, allow_nan=False))
    temp.replace(path)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=serial).encode()).hexdigest()

def array_hash(x):
    return hashlib.sha256(np.ascontiguousarray(x).tobytes()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 ** 2), b''):
            h.update(block)
    return h.hexdigest()

def seed_all(seed):
    """Set random seeds before constructing and training a model."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def metrics(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if len(y) != len(p) or not len(y) or (not np.isfinite(p).all()):
        raise ValueError('Empty, nonfinite or unaligned predictions')
    if not np.isin(y, [0, 1]).all():
        raise ValueError('Unknown labels cannot be scored')
    return dict(n=len(y), positives=int(y.sum()), auc=float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None, ap=float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else None)

def profiles(mode):
    """Use a small budget for a smoke test or the report budget for full experiments."""
    if mode not in ['quick', 'report']:
        raise ValueError(mode)
    return dict(mode=mode, seeds=[42] if mode == 'quick' else [17, 42, 73], cutoffs=[21] if mode == 'quick' else [7, 10, 14, 17, 21], max_epochs=2 if mode == 'quick' else 100, patience=1 if mode == 'quick' else 10, trees=30 if mode == 'quick' else None, batch_size=256)

class Dataset:
    """Read the four CSV files and build the transaction graph."""

    def __init__(self, data_dir, expected_competition=True):
        self.data_dir = Path(data_dir)
        self.train = pd.read_csv(self.data_dir / 'train.csv')
        self.test = pd.read_csv(self.data_dir / 'test.csv')
        self.sample = pd.read_csv(self.data_dir / 'sample_submission.csv')
        raw_edges = pd.read_csv(self.data_dir / 'txs_edgelist.csv')
        self.duplicate_edges = int(raw_edges.duplicated().sum())
        self.edges = raw_edges.drop_duplicates(['txId1', 'txId2']).reset_index(drop=True)
        if list(self.train.columns) != ['txId', 'time_step', *FEATURES, 'label']:
            raise ValueError('Unexpected train columns/order')
        if list(self.test.columns) != ['index', 'txId', 'time_step', *FEATURES]:
            raise ValueError('Unexpected test columns/order')
        if list(self.sample.columns) != ['index', 'target']:
            raise ValueError('Submission schema')
        if self.train.txId.duplicated().any() or self.test.txId.duplicated().any():
            raise ValueError('Duplicate IDs')
        if set(self.train.txId) & set(self.test.txId):
            raise ValueError('Overlapping IDs')
        if not np.array_equal(self.sample['index'], self.test['index']):
            raise ValueError('Submission order')
        if not self.test['index'].is_unique:
            raise ValueError('Duplicate submission IDs')
        if not self.train.label.dropna().isin([0, 1]).all():
            raise ValueError('Labels must be 0/1/blank')
        for df in [self.train, self.test]:
            if not np.isfinite(df[FEATURES].to_numpy()).all():
                raise ValueError('Nonfinite raw features')
        if expected_competition:
            assert (len(self.train), len(self.test), len(self.edges)) == (141772, 15329, 234355)
            assert self.train.time_step.between(1, 35).all() and self.test.time_step.between(36, 49).all()
        self.ids = np.unique(np.r_[self.train.txId, self.test.txId, self.edges.to_numpy().ravel()])
        self.src = np.searchsorted(self.ids, self.edges.txId1)
        self.dst = np.searchsorted(self.ids, self.edges.txId2)
        n = len(self.ids)
        self.outgoing = coo_matrix((np.ones(len(self.src)), (self.src, self.dst)), shape=(n, n)).tocsr()
        self.incoming = self.outgoing.T.tocsr()
        self.indeg = np.diff(self.incoming.indptr)
        self.outdeg = np.diff(self.outgoing.indptr)
        self.nc, self.components = connected_components(self.outgoing, directed=False)
        all_rows = pd.concat([self.train, self.test], ignore_index=True).set_index('txId').reindex(self.ids)
        self.node_raw = all_rows[FEATURES].to_numpy(np.float32)
        self.has_features = np.isfinite(self.node_raw).all(1)
        self.node_y = all_rows.label.fillna(-1).to_numpy(np.int64)
        observed = all_rows.time_step.to_numpy(float)
        self.steps = np.full(n, -1, np.int64)
        for c in range(self.nc):
            member = self.components == c
            times = np.unique(observed[member & np.isfinite(observed)])
            if len(times) != 1:
                raise ValueError('Component does not identify exactly one snapshot')
            self.steps[member] = int(times[0])
        if np.any(self.steps[self.src] != self.steps[self.dst]):
            raise ValueError('Cross-snapshot edge')
        q = deque(np.flatnonzero(self.indeg == 0).tolist())
        deg = self.indeg.copy()
        self.order = []
        while q:
            v = q.popleft()
            self.order.append(v)
            for w in self.outgoing.indices[self.outgoing.indptr[v]:self.outgoing.indptr[v + 1]]:
                deg[w] -= 1
                if deg[w] == 0:
                    q.append(int(w))
        if len(self.order) != n:
            raise ValueError('Graph is not a DAG')
        # Unknown labels are graph context, not negative examples.
        self.frame = self.train.loc[self.train.label.notna()].copy().reset_index(drop=True)
        self.frame.label = self.frame.label.astype(np.int64)
        self.pos = np.searchsorted(self.ids, self.frame.txId)
        self.testpos = np.searchsorted(self.ids, self.test.txId)
        self.node_x = np.column_stack([self.node_raw, self.steps, self.indeg, self.outdeg]).astype(np.float32)
        self.raw = self.node_x[self.pos]
        self.y = self.frame.label.to_numpy(np.int64)
        self._extras = {}
        self.data_hashes = {name: file_hash(self.data_dir / name) for name in ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']}
        self.identity = fingerprint(self.data_hashes)

    def split(self, cut, horizon=14):
        t = self.frame.time_step.to_numpy()
        a = np.flatnonzero(t <= cut)
        b = np.flatnonzero((t > cut) & (t <= cut + horizon))
        if not len(a) or not len(b):
            raise ValueError('Empty temporal split')
        return (a, b)


In [ ]:
def prepare_num(A, B, scaler='quantile'):
    """Fit the imputer, constant-column mask and scaler using training rows only."""
    # Fit every transformation using the current training fold.
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    X = imputer.fit_transform(A).astype(np.float32)
    V = imputer.transform(B).astype(np.float32)
    keep = np.ptp(X, axis=0) > 0
    if not keep.any():
        raise ValueError('All numerical inputs constant')
    X = X[:, keep]
    V = V[:, keep]
    if scaler == 'quantile':
        transform = QuantileTransformer(n_quantiles=max(10, min(len(X) // 30, 1000)), output_distribution='normal', subsample=10 ** 9, random_state=0)
        noise = np.random.default_rng(0).normal(0, 1e-05, X.shape).astype(np.float32)
        transform.fit(X + noise)
    elif scaler == 'robust':
        transform = RobustScaler().fit(X)
    elif scaler == 'standard':
        transform = StandardScaler().fit(X)
    elif scaler == 'none':
        transform = None
    else:
        raise ValueError(scaler)
    bundle = (imputer, keep, transform)
    return (apply_num(A, bundle), apply_num(B, bundle), bundle)

def apply_num(X, bundle):
    """Apply a saved numeric transformation without refitting it."""
    imputer, keep, transform = bundle
    X = imputer.transform(X).astype(np.float32)[:, keep]
    return (transform.transform(X) if transform is not None else X).astype(np.float32)

def make_inputs(data, fit, query, variant='raw', scaler='quantile', base='tabm', test=False, genes=None, feature_plan=None):
    """Fit numeric preprocessing on training rows and apply it to the requested rows."""
    if variant != 'raw':
        raise ValueError('This notebook uses the original input features only')
    columns = 168 if base == 'tabm' else 166
    A = data.raw[fit, :columns].copy()
    B = (data.node_x[data.testpos] if test else data.raw[query])[:, :columns].copy()
    X, V, prep = prepare_num(A, B, scaler)
    C = np.empty((len(X), 0), dtype=np.int64)
    D = np.empty((len(V), 0), dtype=np.int64)
    state = dict(variant=variant, base=base, fit_ids_hash=array_hash(data.frame.txId.to_numpy()[fit]), scaler=scaler, numerical_preprocessor=prep, cardinalities=[])
    return (X, V, C, D, state)

def tree_model(name, y, seed, profile):
    """Construct the selected tree or linear baseline with fixed parameters."""
    w = float((y == 0).sum() / max((y == 1).sum(), 1))
    small = profile['trees']
    if name in ['extra_trees']:
        cls = ExtraTreesClassifier
        return cls(n_estimators=small or 500, min_samples_leaf=2, max_features='sqrt', class_weight='balanced', n_jobs=2, random_state=seed)
    if name == 'xgboost':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=small or 1200, learning_rate=0.025, max_depth=6, min_child_weight=5, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1, scale_pos_weight=w, tree_method='hist', n_jobs=2, random_state=seed)
    if name == 'lightgbm':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=small or 600, learning_rate=0.025, num_leaves=31, min_child_samples=40, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1, scale_pos_weight=w, n_jobs=2, random_state=seed, verbosity=-1)
    raise ValueError(name)

def build_net(name, X, cardinalities, config):
    """Construct a neural model from random initial weights."""
    import tabm, rtdl_num_embeddings as emb
    cfg = dict(config)
    cfg.pop('learning_rate')
    cfg.pop('weight_decay')
    bins = emb.compute_bins(torch.from_numpy(X), n_bins=cfg.pop('n_bins'))
    module = emb.PiecewiseLinearEmbeddings(bins, cfg.pop('d_embedding'), activation=False, version='B')
    return tabm.TabM.make(n_num_features=X.shape[1], cat_cardinalities=cardinalities, d_out=2, num_embeddings=module, **cfg).to(DEVICE)

def forward_net(net, name, x, c, pool, y, idx=None):
    return net(x, c if c.shape[1] else None)

@torch.inference_mode()
def net_predict(net, name, V, D, pool, y, batch=256):
    """Run inference and average the 32 member probabilities for TabM."""
    net.eval()
    parts = []
    for start in range(0, len(V), batch):
        x = torch.as_tensor(V[start:start + batch], device=DEVICE)
        c = torch.as_tensor(D[start:start + batch], device=DEVICE)
        logits = forward_net(net, name, x, c, pool, y)
        p = logits.softmax(-1)[..., 1].mean(1)
        parts.append(p.cpu().numpy())
    return np.concatenate(parts)

def predict_tree(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X)
    return model.predict(X)

def split_indices(data, cut, horizon=14, kind='time', split_seed=20261016, inner_horizon=3):
    """Keep the epoch-selection split inside the outer training history."""
    if kind == 'time':
        a, b = data.split(cut, horizon)
        innercut = cut - min(inner_horizon, cut - 4)
        ia, ib = data.split(innercut, cut - innercut)
    elif kind == 'random':
        a, b = train_test_split(np.arange(len(data.frame)), test_size=0.3, stratify=data.y, random_state=split_seed)
        ia, ib = train_test_split(a, test_size=0.3, stratify=data.y[a], random_state=split_seed + 1)
    elif kind == 'component':
        groups = data.components[data.pos]
        ix = np.arange(len(data.frame))
        a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=split_seed).split(ix, groups=groups))
        x, z = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=split_seed + 1).split(a, groups=groups[a]))
        ia, ib = (a[x], a[z])
    else:
        raise ValueError(kind)
    for z in [a, b, ia, ib]:
        if len(np.unique(data.y[z])) < 2:
            raise ValueError('Split needs both labels')
    assert not np.intersect1d(a, b).size and set(ia) | set(ib) <= set(a)
    return tuple((np.sort(z) for z in [a, b, ia, ib]))

FINAL_ENGINE_HASH = '3661498eb4e7e545253aeae84b480c6d4c349dc7e125920bf97989a5e96c0840'

STACK_NAMES = ['tabm', 'lightgbm', 'xgboost', 'extra_trees']

def submission_audit(data, p, path):
    """Check row order and probability values before saving a submission."""
    p = np.asarray(p, dtype=np.float64)
    if p.shape != (len(data.test),) or not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError('Submission probabilities or row count invalid')
    frame = pd.DataFrame({'index': data.sample['index'].to_numpy(), 'target': p})
    if not frame['index'].is_unique:
        raise ValueError('Duplicate submission index')
    np.testing.assert_array_equal(frame['index'], data.test['index'])
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    check = pd.read_csv(path, float_precision='round_trip')
    assert list(check.columns) == ['index', 'target']
    np.testing.assert_array_equal(check['index'], frame['index'])
    np.testing.assert_array_equal(check.target, p)
    return dict(rows=len(frame), columns=list(frame.columns), sha256=file_hash(path), test_ids_hash=array_hash(data.test.txId.to_numpy()), index_matches_test=True, finite_probability_range=True, this_run_kaggle_auc=None, uploaded_to_kaggle=False)

def stack_profile(mode):
    """The actual .95173 pipeline was one seed, max20/patience5, inner3; not long_inner."""
    p = profiles('quick' if mode == 'quick' else 'report')
    p.update(seeds=[42], max_epochs=2 if mode == 'quick' else 20, patience=1 if mode == 'quick' else 5)
    p['protocol'] = 'source_backed_095173_stacking'
    return p

def stack_head_loss(logits, labels):
    losses = F.cross_entropy(logits.flatten(0, 1), labels.repeat_interleave(logits.shape[1]), reduction='none').reshape(len(labels), -1).mean(1)
    return (losses * torch.ones_like(losses)).mean()

def fit_stack_tabm(seed, X, C, y, V, D, profile, epochs=None, val_y=None):
    seed_all(seed)
    config = dict(TABM)
    model = build_net('tabm', X, [], config)
    x = torch.as_tensor(X, device=DEVICE)
    c = torch.as_tensor(C, device=DEVICE)
    target = torch.as_tensor(np.asarray(y).copy(), device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.0003)
    curve = []
    selected = 0
    best = -np.inf
    stale = 0
    for epoch in range(1, (epochs if epochs is not None else profile['max_epochs']) + 1):
        model.train()
        total = 0.0
        for idx in torch.randperm(len(x), device=DEVICE).split(profile['batch_size']):
            opt.zero_grad(set_to_none=True)
            logits = model(x[idx], None)
            loss = stack_head_loss(logits, target[idx])
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite TabM stack loss')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.detach()) * len(idx)
        row = dict(epoch=epoch, loss=total / len(x))
        if epochs is None:
            if val_y is None:
                raise ValueError('Internal validation labels required')
            row.update(metrics(val_y, net_predict(model, 'tabm', V, D, x, target)))
            if row['auc'] is None:
                raise ValueError('Single-class internal validation')
            if row['auc'] > best:
                best = row['auc']
                selected = epoch
                stale = 0
            else:
                stale += 1
        else:
            selected = epoch
        curve.append(row)
        print('Stack TabM seed', seed, 'select' if epochs is None else 'refit', epoch, row.get('auc'), flush=True)
        if epochs is None and stale >= profile['patience']:
            break
    prediction = net_predict(model, 'tabm', V, D, x, target)
    state = dict(state_dict={k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, config=config, cardinalities=[], epochs=selected, seed=seed)
    del model, opt, x, c, target
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (prediction, selected, curve, state)


In [ ]:
def stack_trial(data, out, model, cut, horizon, profile):
    """Train one base model and save predictions for a future OOF block."""
    if model not in STACK_NAMES:
        raise ValueError(model)
    spec = dict(data=data.identity, model=model, cut=cut, horizon=horizon, profile=profile, seed=42, tabm_config=dict(TABM), inner_horizon=3, engine_hash=FINAL_ENGINE_HASH)
    job = Path(out) / 'stack_members' / f'{model}_{fingerprint(spec)[:20]}'
    job.mkdir(parents=True, exist_ok=True)
    if (job / 'complete.json').exists():
        record = json.loads((job / 'complete.json').read_text())
        assert record['spec'] == json.loads(json.dumps(spec))
        assert record['prediction_hash'] == file_hash(job / 'predictions.csv')
        return (pd.read_csv(job / 'predictions.csv', float_precision='round_trip'), record)
    a, b, ia, ib = split_indices(data, cut, horizon, inner_horizon=3)
    selection = []
    curve = []
    epoch = None
    if model == 'tabm':
        A, B, C, D, inner = make_inputs(data, ia, ib)
        _, epoch, selection, _ = fit_stack_tabm(42, A, C, data.y[ia], B, D, profile, val_y=data.y[ib])
        X, V, C, D, outer = make_inputs(data, a, b)
        p, _, curve, state = fit_stack_tabm(42, X, C, data.y[a], V, D, profile, epochs=epoch)
        torch.save(state, job / 'model.pt')
        joblib.dump(dict(inner=inner, outer=outer), job / 'preprocessing.joblib')
    else:
        X, V, C, D, prep = make_inputs(data, a, b, base='table', scaler='none')
        estimator = tree_model(model, data.y[a], 42, profile).fit(X, data.y[a])
        p = predict_tree(estimator, V)
        joblib.dump(dict(estimator=estimator, preprocessing=prep), job / 'model.joblib')
    frame = data.frame.iloc[b][['txId', 'time_step', 'label']].copy()
    frame['prediction'] = np.asarray(p, dtype=float)
    frame.to_csv(job / 'predictions.csv', index=False)
    save_json(job / 'selection.json', selection)
    save_json(job / 'refit_curve.json', curve)
    record = dict(spec=spec, metrics=metrics(frame.label, p), selected_epochs=epoch, path=str(job.resolve()), prediction_hash=file_hash(job / 'predictions.csv'), train_rows=len(a), outer_labels_used_to_fit=False)
    save_json(job / 'complete.json', record)
    return (frame, record)

def stack_matrix(data, out, cut, horizon, profile):
    """Align base-model prediction columns by transaction ID."""
    reference = None
    parts = []
    records = []
    for name in STACK_NAMES:
        frame, record = stack_trial(data, out, name, cut, horizon, profile)
        if reference is None:
            reference = frame[['txId', 'time_step', 'label']].copy()
        else:
            np.testing.assert_array_equal(reference, frame[['txId', 'time_step', 'label']])
        parts.append(frame.prediction.to_numpy())
        records.append(record)
    return (reference, np.column_stack(parts), records)

def clipped_logit(p):
    p = np.clip(np.asarray(p), 1e-06, 1 - 1e-06)
    return np.log(p) - np.log1p(-p)

def train_meta(M, y):
    """Fit the logit scaler and logistic stacker on OOF predictions only."""
    scaler = StandardScaler().fit(clipped_logit(M))
    model = LogisticRegression(C=0.1, max_iter=2000, random_state=42).fit(scaler.transform(clipped_logit(M)), y)
    return dict(scaler=scaler, model=model, names=STACK_NAMES)

def meta_predict(meta, M):
    return meta['model'].predict_proba(meta['scaler'].transform(clipped_logit(M)))[:, 1]

def historical_stack_validation(data, out, profile):
    """Train the stacker on earlier OOF blocks and score periods 22 to 35."""
    frames = []
    matrices = []
    for cut in [7, 14]:
        frame, matrix, _ = stack_matrix(data, out, cut, 7, profile)
        frames.append(frame)
        matrices.append(matrix)
    meta_frame = pd.concat(frames, ignore_index=True)
    M = np.vstack(matrices)
    target, Q, _ = stack_matrix(data, out, 21, 14, profile)
    assert meta_frame.txId.is_unique and meta_frame.time_step.max() <= 21 and (target.time_step.min() > 21)
    assert not set(meta_frame.txId) & set(target.txId)
    # Fit the meta-model on out-of-fold predictions only.
    meta = train_meta(M, meta_frame.label)
    p = meta_predict(meta, Q)
    job = Path(out) / 'stack_validation'
    job.mkdir(parents=True, exist_ok=True)
    joblib.dump(meta, job / 'meta.joblib')
    meta_frame.assign(**{n: M[:, i] for i, n in enumerate(STACK_NAMES)}).to_csv(job / 'meta_oof.csv', index=False)
    target.assign(**{n: Q[:, i] for i, n in enumerate(STACK_NAMES)}, stacking=p).to_csv(job / 'predictions.csv', index=False)
    rows = [dict(model=name, **metrics(target.label, Q[:, i])) for i, name in enumerate(STACK_NAMES)]
    rows.append(dict(model='TabM_tree_stacking', **metrics(target.label, p)))
    save_json(job / 'audit.json', dict(meta_rows=len(meta_frame), meta_periods=[8, 21], outer_periods=[22, 35], duplicate_oof_ids=0, outer_labels_used_to_fit_meta=False, profile=profile, engine_hash=FINAL_ENGINE_HASH))
    pd.DataFrame(rows).to_csv(job / 'metrics.csv', index=False)
    return pd.DataFrame(rows)

def full_stack_member(data, out, model, profile):
    """Refit one base model on all known labels and predict the official test rows."""
    spec = dict(data=data.identity, model=model, profile=profile, seed=42, inner_horizon=3, train_period=[1, 35], engine_hash=FINAL_ENGINE_HASH)
    job = Path(out) / 'stack_full_models' / f'{model}_{fingerprint(spec)[:20]}'
    job.mkdir(parents=True, exist_ok=True)
    if (job / 'complete.json').exists():
        record = json.loads((job / 'complete.json').read_text())
        assert record['spec'] == json.loads(json.dumps(spec))
        assert record['prediction_hash'] == file_hash(job / 'test_prediction.npy')
        assert record['test_ids_hash'] == array_hash(data.test.txId.to_numpy())
        return (np.load(job / 'test_prediction.npy'), record)
    a = np.arange(len(data.frame))
    empty = np.empty(0, dtype=int)
    selection = []
    curve = []
    epoch = None
    if model == 'tabm':
        ia, ib = data.split(32, 3)
        A, B, C, D, inner = make_inputs(data, ia, ib)
        _, epoch, selection, _ = fit_stack_tabm(42, A, C, data.y[ia], B, D, profile, val_y=data.y[ib])
        save_json(job / 'selection.json', selection)
        X, V, C, D, outer = make_inputs(data, a, empty, test=True)
        p, _, curve, state = fit_stack_tabm(42, X, C, data.y, V, D, profile, epochs=epoch)
        torch.save(state, job / 'model.pt')
        joblib.dump(dict(inner=inner, outer=outer), job / 'preprocessing.joblib')
    else:
        X, V, C, D, prep = make_inputs(data, a, empty, test=True, base='table', scaler='none')
        estimator = tree_model(model, data.y, 42, profile).fit(X, data.y)
        p = predict_tree(estimator, V)
        joblib.dump(dict(estimator=estimator, preprocessing=prep), job / 'model.joblib')
    np.save(job / 'test_prediction.npy', p)
    save_json(job / 'refit_curve.json', curve)
    record = dict(spec=spec, selected_epochs=epoch, train_rows=len(a), unknown_rows_used_as_negative=0, test_ids_hash=array_hash(data.test.txId.to_numpy()), prediction_hash=file_hash(job / 'test_prediction.npy'), path=str(job.resolve()))
    save_json(job / 'complete.json', record)
    return (p, record)

def final_stack_submission(data, out, profile):
    """Fit the stacker on unique forward OOF rows, then refit the base models."""
    frames = []
    matrices = []
    for cut in [7, 14, 21, 28]:
        frame, matrix, _ = stack_matrix(data, out, cut, 7, profile)
        frames.append(frame)
        matrices.append(matrix)
    meta_frame = pd.concat(frames, ignore_index=True)
    M = np.vstack(matrices)
    assert meta_frame.txId.is_unique and meta_frame.time_step.between(8, 35).all()
    assert set(meta_frame.txId) == set(data.frame.loc[data.frame.time_step >= 8, 'txId'])
    assert not set(meta_frame.txId) & set(data.test.txId)
    job = Path(out) / 'final_stack'
    job.mkdir(parents=True, exist_ok=True)
    # Fit the meta-model on out-of-fold predictions only.
    meta = train_meta(M, meta_frame.label)
    joblib.dump(meta, job / 'meta.joblib')
    meta_frame.assign(**{n: M[:, i] for i, n in enumerate(STACK_NAMES)}).to_csv(job / 'meta_oof.csv', index=False)
    parts = []
    records = []
    for name in STACK_NAMES:
        p, record = full_stack_member(data, out, name, profile)
        parts.append(p)
        records.append(record)
    Q = np.column_stack(parts)
    p = meta_predict(meta, Q)
    pd.DataFrame({'index': data.test['index'], **{n: Q[:, i] for i, n in enumerate(STACK_NAMES)}}).to_csv(job / 'test_base_predictions.csv', index=False)
    filename = 'QUICK_CHECK_ONLY_stacking.csv' if profile['mode'] == 'quick' else 'submission_tabm_tree_stacking.csv'
    path = job / filename
    audit = submission_audit(data, p, path)
    audit.update(profile=profile, engine_hash=FINAL_ENGINE_HASH, model_names=STACK_NAMES, seed=42, tabm_dropout=0.1, inner_horizon=3, meta_C=0.1, meta_oof_periods=[[8, 14], [15, 21], [22, 28], [29, 35]], meta_rows=len(meta_frame), meta_duplicate_ids=0, outer_labels_used_to_fit_meta=False, test_labels_used=False, meta_coefficients=meta['model'].coef_.ravel(), meta_intercept=meta['model'].intercept_, full_model_records=records, reported_public_auc=0.95173, reported_public_source='User-scored source pipeline; not a score computed in this run', eligible_full_run=profile['mode'] == 'report')
    save_json(job / 'submission_audit.json', audit)
    return (path, audit)


In [ ]:
P = profiles(PROFILE)


## 3. Load and check the data


In [ ]:
# Labelled rows are the only rows used by supervised models.
data = Dataset(DATA_DIR, expected_competition=EXPECTED_COMPETITION)
print('Labelled training rows:', len(data.frame), '| Unknown labels:', int(data.train.label.isna().sum()), '| Test rows:', len(data.test))


## 4. Fixed stacking recipe
The traceable 0.95173 pipeline uses seed 42. TabM has dropout 0.1, a three-period inner window, at most 20 epochs and patience 5. LightGBM/XGBoost/ExtraTrees use 600/1200/500 trees. TabM uses degree features; the trees use the 165 attributes plus time_step.

This historical stacking pipeline uses the original TabM selection policy, rather than the seven-period policy in notebook 06. Its original submission CSV is included separately in the package.


In [ ]:
SP = stack_profile(PROFILE)
print('Stacking profile:', SP)


## 5. Optional outer validation
Fit the meta-model on OOF periods 8-14 and 15-21; evaluate on periods 22-35. Outer labels do not train the meta-model. This extra validation is optional because the final submission requires its own four-block OOF set.


In [ ]:
RUN_STACK_VALIDATION = globals().get('RUN_STACK_VALIDATION', False)
if RUN_STACK_VALIDATION:
    display(historical_stack_validation(data, OUT, SP))
else:
    print('Optional outer stacking validation skipped.')


## 6. OOF training, meta-model and full refit
Create four non-overlapping OOF blocks: train through 7/14/21/28 and predict 8-14/15-21/22-28/29-35. Each transaction appears once. Clip base probabilities, take logits and fit a StandardScaler plus LogisticRegression with C=0.1 using OOF rows only. Finally refit each base model on all known labels, predict test and apply the meta-model.


In [ ]:
submission, stack_audit = final_stack_submission(data, OUT, SP)
print('Unique OOF rows:', stack_audit['meta_rows'])
print('Models:', stack_audit['model_names'])


## 7. Check and download the submission
The historical Public AUC is 0.95173. This run does not calculate a new Kaggle score. Weights, OOF predictions, base predictions and the submission audit are saved with the output.


In [ ]:
frame = pd.read_csv(submission, float_precision='round_trip')
display(frame.head())
print({key: stack_audit[key] for key in ['rows','index_matches_test','finite_probability_range','this_run_kaggle_auc']})
print('SHA256:', file_hash(submission))


In [ ]:
DOWNLOAD_CSV = globals().get('DOWNLOAD_CSV', False)
if DOWNLOAD_CSV:
    from google.colab import files
    files.download(str(submission))
else:
    print('Saved to:', submission)
